# Lightweight Machine Learning-Based Network Intrusion Detection for Resource-Constrained Edge Devices

## Project Overview

This notebook implements and evaluates a lightweight machine learning-based Network Intrusion Detection System (NIDS) using the Edge-IIoTset dataset.

### Research Objective
To evaluate the trade-off between network attack detection performance and computational resource consumption when using lightweight machine learning approaches.

### Dataset
Edge-IIoTset:
- ML-EdgeIIoT-dataset.csv
- Individual attack and normal traffic CSV files where required.

### Evaluation Environment
- Hardware: MacBook Pro
- Language: Python
- Libraries: Pandas, NumPy, Scikit-learn and Matplotlib.

### Research Scope
The study focuses on controlled host-based evaluation as a proxy for resource-constrained edge deployment. It does not claim physical edge-device validation.

In [3]:
import os
import sys
import platform

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)

Python: 3.12.5 (v3.12.5:ff3bc82f7c9, Aug  7 2024, 05:32:06) [Clang 13.0.0 (clang-1300.0.29.30)]
Platform: macOS-13.7.8-x86_64-i386-64bit
Pandas: 3.0.2
NumPy: 1.26.4
Scikit-learn: 1.5.1


## 1.1 Dataset Configuration

The study uses the Edge-IIoTset dataset. The main dataset selected for machine learning experiments is ML-EdgeIIoT-dataset.csv.

The individual attack and normal traffic CSV files will only be examined where necessary for dataset verification and evaluation design.

In [4]:
# Define the dataset location
DATASET_PATH = os.path.join(
    "Edge-IIoTset dataset",
    "Selected dataset for ML and DL",
    "ML-EdgeIIoT-dataset.csv"
)

# Verify that the dataset exists
if os.path.exists(DATASET_PATH):
    print("Dataset found:", DATASET_PATH)
    print("File size:", round(os.path.getsize(DATASET_PATH) / (1024 * 1024), 2), "MB")
else:
    print("Dataset not found. Check the project directory.")

Dataset found: Edge-IIoTset dataset/Selected dataset for ML and DL/ML-EdgeIIoT-dataset.csv
File size: 78.38 MB


## 2. Dataset Inspection

This section examines the structure of the selected Edge-IIoTset dataset before any preprocessing or model training.

The inspection focuses on:
- Dataset dimensions.
- Feature names and data types.
- Target labels.
- Potential capture or session identifiers.
- Initial sample records.

In [5]:
# Load a small sample for initial inspection
sample_df = pd.read_csv(DATASET_PATH, nrows=5)

# Display dataset structure
print("Number of columns:", len(sample_df.columns))

print("\nColumn names:")
for column in sample_df.columns:
    print("-", column)

print("\nData types:")
display(sample_df.dtypes.to_frame(name="Data Type"))

print("\nFirst five records:")
display(sample_df)

Number of columns: 63

Column names:
- frame.time
- ip.src_host
- ip.dst_host
- arp.dst.proto_ipv4
- arp.opcode
- arp.hw.size
- arp.src.proto_ipv4
- icmp.checksum
- icmp.seq_le
- icmp.transmit_timestamp
- icmp.unused
- http.file_data
- http.content_length
- http.request.uri.query
- http.request.method
- http.referer
- http.request.full_uri
- http.request.version
- http.response
- http.tls_port
- tcp.ack
- tcp.ack_raw
- tcp.checksum
- tcp.connection.fin
- tcp.connection.rst
- tcp.connection.syn
- tcp.connection.synack
- tcp.dstport
- tcp.flags
- tcp.flags.ack
- tcp.len
- tcp.options
- tcp.payload
- tcp.seq
- tcp.srcport
- udp.port
- udp.stream
- udp.time_delta
- dns.qry.name
- dns.qry.name.len
- dns.qry.qu
- dns.qry.type
- dns.retransmission
- dns.retransmit_request
- dns.retransmit_request_in
- mqtt.conack.flags
- mqtt.conflag.cleansess
- mqtt.conflags
- mqtt.hdrflags
- mqtt.len
- mqtt.msg_decoded_as
- mqtt.msg
- mqtt.msgtype
- mqtt.proto_len
- mqtt.protoname
- mqtt.topic
- mqtt.topic_

,Data Type
frame.time,float64
ip.src_host,str
ip.dst_host,float64
arp.dst.proto_ipv4,float64
arp.opcode,float64
...,...
mbtcp.len,float64
mbtcp.trans_id,float64
mbtcp.unit_id,float64
Attack_label,int64



First five records:


,frame.time,ip.src_host,ip.dst_host,arp.dst.proto_ipv4,arp.opcode,arp.hw.size,arp.src.proto_ipv4,icmp.checksum,icmp.seq_le,icmp.transmit_timestamp,...,mqtt.proto_len,mqtt.protoname,mqtt.topic,mqtt.topic_len,mqtt.ver,mbtcp.len,mbtcp.trans_id,mbtcp.unit_id,Attack_label,Attack_type
0,6.0,192.168.0.152,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
1,6.0,192.168.0.101,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
2,6.0,192.168.0.152,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
3,6.0,192.168.0.101,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
4,6.0,192.168.0.152,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM


In [6]:
# Count records using the CSV file structure
with open(DATASET_PATH, "r", encoding="utf-8") as file:
    total_rows = sum(1 for _ in file) - 1

print("Total records:", f"{total_rows:,}")
print("Total features:", len(sample_df.columns))

Total records: 157,800
Total features: 63


## 3. Target Variable and Data Type Inspection

This section examines the target variables, `Attack_label` and `Attack_type`, to understand how normal and malicious traffic is represented. It also checks the data types of the dataset fields before preprocessing.

In [7]:
# Inspect data types
display(sample_df.dtypes.to_frame(name="Data Type"))

# Load only the target columns
target_df = pd.read_csv(
    DATASET_PATH,
    usecols=["Attack_label", "Attack_type"]
)

# Examine target values
print("Attack_label distribution:")
display(target_df["Attack_label"].value_counts(dropna=False))

print("\nAttack_type distribution:")
display(target_df["Attack_type"].value_counts(dropna=False))

# Check missing target values
print("\nMissing target values:")
display(target_df[["Attack_label", "Attack_type"]].isnull().sum())

,Data Type
frame.time,float64
ip.src_host,str
ip.dst_host,float64
arp.dst.proto_ipv4,float64
arp.opcode,float64
...,...
mbtcp.len,float64
mbtcp.trans_id,float64
mbtcp.unit_id,float64
Attack_label,int64


Attack_label distribution:


Attack_label
1    133499
0     24301
Name: count, dtype: int64


Attack_type distribution:


Attack_type
Normal                   24301
DDoS_UDP                 14498
DDoS_ICMP                14090
Ransomware               10925
DDoS_HTTP                10561
SQL_injection            10311
Uploading                10269
DDoS_TCP                 10247
Backdoor                 10195
Vulnerability_scanner    10076
Port_Scanning            10071
XSS                      10052
Password                  9989
MITM                      1214
Fingerprinting            1001
Name: count, dtype: int64


Missing target values:


Attack_label    0
Attack_type     0
dtype: int64